# Finetuning YOLOv8 no SH17

Notebook de estudo para treino e validacao do dataset SH17.

In [1]:
from pathlib import Path
import yaml
import torch
from ultralytics import YOLO

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent, cwd.parent.parent.parent]
project_root = None
for c in candidates:
    if (c / 'pyproject.toml').exists() and (c / 'data/images/sh17_dataset').exists():
        project_root = c
        break

if project_root is None:
    raise RuntimeError('Nao foi possivel localizar project_root automaticamente.')

data_yaml = project_root / 'data/images/sh17_dataset/data.yaml'
model_path = project_root / 'notebooks/yolov8n.pt'

print('project_root =', project_root)
print('data_yaml exists =', data_yaml.exists())
print('model exists =', model_path.exists())
print('cuda available =', torch.cuda.is_available())
print('cuda device count =', torch.cuda.device_count())

project_root = /home/senacgoon.local/202473567/Projects/senac_ia_uc_16_computer_vision
data_yaml exists = True
model exists = True
cuda available = True
cuda device count = 1


In [2]:
with open(data_yaml, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

print('nc =', cfg['nc'])
for idx, name in cfg['names'].items():
    print(idx, name)

nc = 17
0 person
1 ear
2 ear-mufs
3 face
4 face-guard
5 face-mask-medical
6 foot
7 tools
8 glasses
9 gloves
10 helmet
11 hands
12 head
13 medical-suit
14 shoes
15 safety-suit
16 safety-vest


In [3]:
dataset_root = (project_root / 'data/images/sh17_dataset').resolve()
print('dataset_root =', dataset_root)
print('train_files exists =', (dataset_root / 'train_files.txt').exists())
print('val_files exists =', (dataset_root / 'val_files.txt').exists())

train_preview = (dataset_root / 'train_files.txt').read_text(encoding='utf-8').splitlines()[:3]
print('preview train original =', train_preview)
print('preview train expected =', [f"images/{x}" for x in train_preview])

dataset_root = /home/senacgoon.local/202473567/Projects/senac_ia_uc_16_computer_vision/data/images/sh17_dataset
train_files exists = True
val_files exists = True
preview train original = ['pexels-photo-16468065.jpeg', 'pexels-photo-4219650.jpeg', 'pexels-photo-8948288.jpeg']
preview train expected = ['images/pexels-photo-16468065.jpeg', 'images/pexels-photo-4219650.jpeg', 'images/pexels-photo-8948288.jpeg']


In [4]:
from ultralytics.data.utils import check_det_dataset

# Gera os mesmos artefatos usados na celula de treino
_runtime_train = dataset_root / 'runtime_train_files.txt'
_runtime_val = dataset_root / 'runtime_val_files.txt'

for src_name, dst in [('train_files.txt', _runtime_train), ('val_files.txt', _runtime_val)]:
    src = dataset_root / src_name
    lines = [ln.strip() for ln in src.read_text(encoding='utf-8').splitlines() if ln.strip()]
    lines = [ln if ln.startswith('images/') else f'images/{ln}' for ln in lines]
    dst.write_text('\n'.join(lines) + '\n', encoding='utf-8')

runtime_cfg = dict(cfg)
runtime_cfg['path'] = str(dataset_root)
runtime_cfg['train'] = _runtime_train.name
runtime_cfg['val'] = _runtime_val.name
runtime_yaml = dataset_root / 'data.runtime.yaml'
with open(runtime_yaml, 'w', encoding='utf-8') as f:
    yaml.safe_dump(runtime_cfg, f, sort_keys=False, allow_unicode=False)

print('runtime_yaml =', runtime_yaml)
print('primeira linha train runtime =', _runtime_train.read_text(encoding='utf-8').splitlines()[0])

checked = check_det_dataset(str(runtime_yaml))
print('check_det_dataset ok')
print('resolved train =', checked['train'])
print('resolved val =', checked['val'])

runtime_yaml = /home/senacgoon.local/202473567/Projects/senac_ia_uc_16_computer_vision/data/images/sh17_dataset/data.runtime.yaml
primeira linha train runtime = images/pexels-photo-16468065.jpeg
check_det_dataset ok
resolved train = /home/senacgoon.local/202473567/Projects/senac_ia_uc_16_computer_vision/data/images/sh17_dataset/runtime_train_files.txt
resolved val = /home/senacgoon.local/202473567/Projects/senac_ia_uc_16_computer_vision/data/images/sh17_dataset/runtime_val_files.txt


In [7]:
model = YOLO(str(model_path))

dataset_root = (project_root / 'data/images/sh17_dataset').resolve()

def _rewrite_split(split_name: str) -> Path:
    src = dataset_root / split_name
    dst = dataset_root / f"runtime_{split_name}"
    lines = [ln.strip() for ln in src.read_text(encoding='utf-8').splitlines() if ln.strip()]
    fixed = []
    for ln in lines:
        # O Ultralytics espera caminhos de imagem; os splits originais trazem apenas nomes.
        fixed.append(ln if ln.startswith('images/') else f"images/{ln}")
    dst.write_text("\n".join(fixed) + "\n", encoding='utf-8')
    return dst

runtime_train = _rewrite_split('train_files.txt')
runtime_val = _rewrite_split('val_files.txt')

data_cfg = dict(cfg)
data_cfg['path'] = str(dataset_root)
data_cfg['train'] = runtime_train.name
data_cfg['val'] = runtime_val.name

tmp_data_yaml = dataset_root / 'data.runtime.yaml'
with open(tmp_data_yaml, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False, allow_unicode=False)

print('Usando dataset em:', dataset_root)
print('Split treino:', runtime_train)
print('Split validacao:', runtime_val)
print('YAML de treino:', tmp_data_yaml)

results = model.train(
    data=str(tmp_data_yaml),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    patience=20,
    project=str(project_root / 'runs/detect'),
    name='sh17_yolov8n_finetune',
)

results

Usando dataset em: /home/senacgoon.local/202473567/Projects/senac_ia_uc_16_computer_vision/data/images/sh17_dataset
Split treino: /home/senacgoon.local/202473567/Projects/senac_ia_uc_16_computer_vision/data/images/sh17_dataset/runtime_train_files.txt
Split validacao: /home/senacgoon.local/202473567/Projects/senac_ia_uc_16_computer_vision/data/images/sh17_dataset/runtime_val_files.txt
YAML de treino: /home/senacgoon.local/202473567/Projects/senac_ia_uc_16_computer_vision/data/images/sh17_dataset/data.runtime.yaml
New https://pypi.org/project/ultralytics/8.4.51 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.49 🚀 Python-3.12.3 torch-2.7.1+cu126 CUDA:0 (NVIDIA L40S-16C, 16151MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, da

Model summary: 130 layers, 3,014,163 parameters, 3,014,147 gradients, 8.2 GFLOPs

Transferred 319/355 items from pretrained weights
Freezing layer 'model.22.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
WARNING ⚠️ train: Image speed checks: failed to access files
train: Scanning images... 0 images, 0 backgrounds, 6479 corrupt: 100% ━━━━━━━━━━━━ 6479/6479 68.4Kit/s 0.1s
train: images/career-firefighter-relaxing-job-162540.jpeg: ignoring corrupt image/label: [Errno 2] No such file or directory: 'images/career-firefighter-relaxing-job-162540.jpeg'
train: images/carrying-head-indian-farm-worker-158011.jpeg: ignoring corrupt image/label: [Errno 2] No such file or directory: 'images/carrying-head-indian-farm-worker-158011.jpeg'
train: images/construction-site-build-construction-work-159306.jpeg: ignoring corrupt image/label: [Errno 2] No such file or directory: 'images/construction-site-build-construction-work-159306.jpeg'
train: images/construc

RuntimeError: No valid images found in images.cache.
  [34m[1mtrain: [0mimages/career-firefighter-relaxing-job-162540.jpeg: ignoring corrupt image/label: [Errno 2] No such file or directory: 'images/career-firefighter-relaxing-job-162540.jpeg'
  [34m[1mtrain: [0mimages/carrying-head-indian-farm-worker-158011.jpeg: ignoring corrupt image/label: [Errno 2] No such file or directory: 'images/carrying-head-indian-farm-worker-158011.jpeg'
  [34m[1mtrain: [0mimages/construction-site-build-construction-work-159306.jpeg: ignoring corrupt image/label: [Errno 2] No such file or directory: 'images/construction-site-build-construction-work-159306.jpeg'
  [34m[1mtrain: [0mimages/construction-site-build-construction-work-159358.jpeg: ignoring corrupt image/label: [Errno 2] No such file or directory: 'images/construction-site-build-construction-work-159358.jpeg'
  [34m[1mtrain: [0mimages/construction-site-build-construction-work-159375.jpeg: ignoring corrupt image/label: [Errno 2] No su

In [6]:
best_model = project_root / 'runs/detect/sh17_yolov8n_finetune/weights/best.pt'
model_best = YOLO(str(best_model))
metrics = model_best.val(data=str(data_yaml), device=0)
metrics

FileNotFoundError: [Errno 2] No such file or directory: '/home/senacgoon.local/202473567/Projects/senac_ia_uc_16_computer_vision/runs/detect/sh17_yolov8n_finetune/weights/best.pt'